
# NMPC Reference Tracking with ACADOS

This notebook builds a clean nonlinear MPC reference-tracking controller using **ACADOS** and `acados_template`.

The example is written for the RC-truck style pitch/longitudinal model:


$$
q = \begin{bmatrix}p & v & \theta & \omega\end{bmatrix}^\top,
\qquad u = \tau
$$

with dynamics

$$
\dot p = v
$$

$$
\dot v = \frac{\tau}{m r} - c_v v
$$

$$
\dot \theta = \omega
$$

$$
\dot \omega = \frac{-\tau + m g l \cos(\theta)}{I_{\mathrm{eff}}}.
$$

The NMPC problem is

```math
\min_{q_{0:N},u_{0:N-1}}
\sum_{k=0}^{N-1}
\left[
a(q_k-q_k^{\mathrm{ref}})^\top Q(q_k-q_k^{\mathrm{ref}})
+ (u_k-u_k^{\mathrm{ref}})^\top R(u_k-u_k^{\mathrm{ref}})
\right]
+ (q_N-q_N^{\mathrm{ref}})^\top Q_f(q_N-q_N^{\mathrm{ref}})
```

subject to

$$
q_0 = q_{\mathrm{meas}},
\qquad q_{k+1} = F(q_k,u_k),
\qquad u_{\min} \le u_k \le u_{\max},
$$

and optional state bounds.

Important: ACADOS and CasADi use **radians**, so all angles inside the solver are radians. Plots below convert angle to degrees only for visualization.



## 0. ACADOS setup reminder

This notebook assumes ACADOS is already compiled and the Python interface is available.

A typical Linux/WSL setup is:

```bash
git clone https://github.com/acados/acados.git
cd acados
git submodule update --recursive --init
mkdir -p build
cd build
cmake -DACADOS_WITH_QPOASES=ON ..
make install -j4

pip install -e <acados_root>/interfaces/acados_template

export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:<acados_root>/lib
export ACADOS_SOURCE_DIR=<acados_root>
```

After that, restart Jupyter from the same terminal where the environment variables are active.


In [59]:
%env ACADOS_SOURCE_DIR=/home/juan/lehigh_PhD/monster_truck/NMPC/acados
%env LD_LIBRARY_PATH=/home/juan/lehigh_PhD/monster_truck/NMPC/acados/lib

env: ACADOS_SOURCE_DIR=/home/juan/lehigh_PhD/monster_truck/NMPC/acados
env: LD_LIBRARY_PATH=/home/juan/lehigh_PhD/monster_truck/NMPC/acados/lib


In [60]:

import os
import shutil
import numpy as np
import matplotlib.pyplot as plt

try:
    from casadi import SX, vertcat, cos
    from acados_template import AcadosModel, AcadosOcp, AcadosOcpSolver
    ACADOS_READY = True
    print("ACADOS imports are ready.")
except Exception as e:
    ACADOS_READY = False
    print("ACADOS is not ready in this Python environment.")
    print("Reason:", repr(e))
    print("Install/build ACADOS first, then restart the notebook kernel.")


ACADOS imports are ready.


In [61]:

from dataclasses import dataclass

@dataclass
class TruckConfig:
    # Physical parameters
    m: float = 5.1
    r: float = 0.081
    l: float = 0.20
    g: float = 9.81
    c_v: float = 9.0
    body_length: float = 0.53
    body_height: float = 0.30

    # NMPC settings
    dt: float = 0.05
    N: int = 15
    sim_time: float = 10.0

    # Bounds
    tau_min: float = -8.0
    tau_max: float = 8.0
    v_min: float = -5.0
    v_max: float = 5.0
    theta_min_deg: float = -90.0
    theta_max_deg: float = 0.0
    omega_min: float = -8.0
    omega_max: float = 8.0

    # Reference angle. Negative can represent nose-up if that is your sign convention.
    theta_ref_deg: float = 0.0

    # Cost weights. These are radian-based angle weights.
    q_p: float = 0.0
    q_v: float = 0.0
    q_theta: float = 40.0
    q_omega: float = 0.0
    r_tau: float = 0.05

    qf_p: float = 0.0
    qf_v: float = 0.0
    qf_theta: float = 100.0
    qf_omega: float = 0.0

    @property
    def I_body(self):
        return (1.0 / 12.0) * self.m * (self.body_length**2 + self.body_height**2)

    @property
    def I_eff(self):
        return self.I_body + self.m * self.l**2

    @property
    def T_horizon(self):
        return self.N * self.dt

cfg = TruckConfig()
print(cfg)
print("I_body =", cfg.I_body)
print("I_eff  =", cfg.I_eff)


TruckConfig(m=5.1, r=0.081, l=0.2, g=9.81, c_v=9.0, body_length=0.53, body_height=0.3, dt=0.05, N=15, sim_time=10.0, tau_min=-8.0, tau_max=8.0, v_min=-5.0, v_max=5.0, theta_min_deg=-90.0, theta_max_deg=0.0, omega_min=-8.0, omega_max=8.0, theta_ref_deg=0.0, q_p=0.0, q_v=0.0, q_theta=40.0, q_omega=0.0, r_tau=0.05, qf_p=0.0, qf_v=0.0, qf_theta=100.0, qf_omega=0.0)
I_body = 0.15763249999999998
I_eff  = 0.3616325



## 1. Reference generator

For this model, a flat pitch equilibrium generally needs a nonzero torque:

$$
\tau_{\mathrm{eq}} = m g l \cos(\theta_{\mathrm{ref}}).$$

At steady speed, the longitudinal equation also gives

$$
v_{\mathrm{eq}} = \frac{\tau_{\mathrm{eq}}}{m r c_v}.
$$

That is why the input reference below is **not zero**. If you set $u^{\mathrm{ref}}=0$ while the model requires nonzero equilibrium torque, the controller will fight the physics of the model.


In [62]:
def reference_at(t, cfg):
    p_ref = 0.0
    v_ref = 0.0
    theta_ref = np.deg2rad(cfg.theta_ref_deg)
    omega_ref = 0.0

    x_ref = np.array([
        p_ref,
        v_ref,
        theta_ref,
        omega_ref,
    ], dtype=float)

    return x_ref


## 2. Define the ACADOS model

ACADOS needs an explicit ODE expression and an implicit ODE expression:

$$
f_{\mathrm{impl}}(\dot q,q,u)=\dot q-f_{\mathrm{expl}}(q,u).
$$


In [63]:

def create_acados_model(cfg):
    model = AcadosModel()
    model.name = "rc_truck_reference_tracking"

    # State: [position, velocity, pitch, pitch_rate]
    p = SX.sym("p")
    v = SX.sym("v")
    theta = SX.sym("theta")
    omega = SX.sym("omega")
    x = vertcat(p, v, theta, omega)

    # Control: torque
    tau = SX.sym("tau")
    u = vertcat(tau)

    # State derivative
    p_dot = SX.sym("p_dot")
    v_dot = SX.sym("v_dot")
    theta_dot = SX.sym("theta_dot")
    omega_dot = SX.sym("omega_dot")
    xdot = vertcat(p_dot, v_dot, theta_dot, omega_dot)

    f_expl = vertcat(
        v,
        tau / (cfg.m * cfg.r) - cfg.c_v * v,
        omega,
        (-tau + cfg.m * cfg.g * cfg.l * cos(theta)) / cfg.I_eff,
    )

    model.x = x
    model.u = u
    model.xdot = xdot
    model.f_expl_expr = f_expl
    model.f_impl_expr = xdot - f_expl

    return model



## 3. Build the ACADOS OCP

This uses a linear least-squares cost:

$$
y_k = \begin{bmatrix}q_k \\ u_k\end{bmatrix},
\qquad y_k^{\mathrm{ref}}=\begin{bmatrix}q_k^{\mathrm{ref}} \\ u_k^{\mathrm{ref}}\end{bmatrix}.
$$

The terminal cost tracks only the terminal state.


In [64]:
def create_ocp_solver(cfg, x0):
    if not ACADOS_READY:
        print("ACADOS is not ready. Solver was not created.")
        return None

    model = create_acados_model(cfg)

    nx = 4
    nu = 1
    ny = nx + nu
    ny_e = nx

    ocp = AcadosOcp()
    ocp.model = model

    ocp.solver_options.N_horizon = cfg.N
    ocp.solver_options.tf = cfg.N * cfg.dt

    Q = np.diag([
        cfg.q_p,
        cfg.q_v,
        cfg.q_theta,
        cfg.q_omega,
    ])

    R = np.diag([
        cfg.r_tau,
    ])

    Qf = np.diag([
        cfg.qf_p,
        cfg.qf_v,
        cfg.qf_theta,
        cfg.qf_omega,
    ])

    ocp.cost.cost_type = "LINEAR_LS"
    ocp.cost.cost_type_e = "LINEAR_LS"

    # Running cost: y_k = [x_k, u_k], dimension 5
    ocp.cost.W = np.zeros((ny, ny))
    ocp.cost.W[:nx, :nx] = Q
    ocp.cost.W[nx:, nx:] = R

    ocp.cost.Vx = np.zeros((ny, nx))
    ocp.cost.Vx[:nx, :] = np.eye(nx)

    ocp.cost.Vu = np.zeros((ny, nu))
    ocp.cost.Vu[nx:, :] = np.eye(nu)

    ocp.cost.yref = np.zeros(ny)

    # Terminal cost: y_N = x_N, dimension 4
    ocp.cost.W_e = Qf
    ocp.cost.Vx_e = np.eye(nx)
    ocp.cost.yref_e = np.zeros(ny_e)

    print("W shape      =", ocp.cost.W.shape)
    print("Vx shape     =", ocp.cost.Vx.shape)
    print("Vu shape     =", ocp.cost.Vu.shape)
    print("yref shape   =", ocp.cost.yref.shape)
    print("W_e shape    =", ocp.cost.W_e.shape)
    print("Vx_e shape   =", ocp.cost.Vx_e.shape)
    print("yref_e shape =", ocp.cost.yref_e.shape)

    ocp.constraints.x0 = np.asarray(x0, dtype=float).reshape(nx)

    ocp.constraints.idxbu = np.array([0])
    ocp.constraints.lbu = np.array([cfg.tau_min])
    ocp.constraints.ubu = np.array([cfg.tau_max])

    ocp.constraints.idxbx = np.array([1, 2, 3])

    ocp.constraints.lbx = np.array([
        cfg.v_min,
        np.deg2rad(cfg.theta_min_deg),
        cfg.omega_min,
    ])

    ocp.constraints.ubx = np.array([
        cfg.v_max,
        np.deg2rad(cfg.theta_max_deg),
        cfg.omega_max,
    ])

    ocp.solver_options.qp_solver = "PARTIAL_CONDENSING_HPIPM"
    ocp.solver_options.hessian_approx = "GAUSS_NEWTON"
    ocp.solver_options.integrator_type = "ERK"
    ocp.solver_options.nlp_solver_type = "SQP_RTI"

    ocp.solver_options.sim_method_num_stages = 4
    ocp.solver_options.sim_method_num_steps = 1

    ocp.solver_options.qp_solver_cond_N = cfg.N
    ocp.solver_options.print_level = 0

    json_file = "acados_ocp_rc_truck_reference_tracking.json"

    solver = AcadosOcpSolver(
        ocp,
        json_file=json_file,
        build=True,
        generate=True,
    )

    return solver


## 4. Plant simulation helper

The real plant can be MuJoCo, ROS, or hardware. Here we simulate the same continuous model with RK4 so the notebook is self-contained.


In [65]:

def dynamics_np(x, u, cfg):
    p, v, theta, omega = x
    tau = float(np.asarray(u).reshape(-1)[0])

    return np.array([
        v,
        tau / (cfg.m * cfg.r) - cfg.c_v * v,
        omega,
        (-tau + cfg.m * cfg.g * cfg.l * np.cos(theta)) / cfg.I_eff,
    ], dtype=float)


def rk4_step(x, u, dt, cfg):
    k1 = dynamics_np(x, u, cfg)
    k2 = dynamics_np(x + 0.5 * dt * k1, u, cfg)
    k3 = dynamics_np(x + 0.5 * dt * k2, u, cfg)
    k4 = dynamics_np(x + dt * k3, u, cfg)
    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)



## 5. Closed-loop NMPC simulation

At each control step:

1. measure the current state,
2. set the initial-state equality constraint,
3. set the reference over the full horizon,
4. solve the OCP,
5. apply only the first input,
6. shift to the next time step.


In [66]:
def set_reference_trajectory(solver, t_now, cfg):
    nx = 4
    nu = 1

    for j in range(cfg.N):
        t_j = t_now + j * cfg.dt

        x_ref_j = reference_at(t_j, cfg)
        x_ref_j = np.asarray(x_ref_j, dtype=float).reshape(nx)

        # Stage cost expects yref = [x_ref, 0]
        # The last zero is not a desired control trajectory.
        # It only makes r_tau * tau^2.
        yref_j = np.concatenate([
            x_ref_j,
            np.zeros(nu),
        ])

        solver.cost_set(j, "yref", yref_j)

    x_ref_N = reference_at(t_now + cfg.N * cfg.dt, cfg)
    x_ref_N = np.asarray(x_ref_N, dtype=float).reshape(nx)

    solver.cost_set(cfg.N, "yref", x_ref_N)


def solve_nmpc_step(solver, x_meas, t_now, cfg):
    # Current-state equality constraint q0 = q_meas
    solver.constraints_set(0, "lbx", x_meas)
    solver.constraints_set(0, "ubx", x_meas)

    set_reference_trajectory(solver, t_now, cfg)

    status = solver.solve()
    if status != 0:
        print(f"ACADOS returned status {status} at t={t_now:.3f} s")

    u0 = solver.get(0, "u")
    return np.array(u0).reshape(1), status


In [67]:
import ctypes

ACADOS_PATH = "/home/juan/lehigh_PhD/monster_truck/NMPC/acados"
lib = ACADOS_PATH + "/lib/"

ctypes.CDLL(lib + "libblasfeo.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(lib + "libhpipm.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(lib + "libqpOASES_e.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL(lib + "libacados.so", mode=ctypes.RTLD_GLOBAL)

print("ACADOS libraries loaded correctly.")

ACADOS libraries loaded correctly.


In [68]:
import ctypes
import os

ACADOS_PATH = "/home/juan/lehigh_PhD/monster_truck/NMPC/acados"

ctypes.CDLL(ACADOS_PATH + "/lib/libqpOASES_e.so")
ctypes.CDLL(ACADOS_PATH + "/lib/libacados.so")

print("ACADOS libraries loaded correctly.")

ACADOS libraries loaded correctly.


In [69]:
import os
import shutil
import glob

shutil.rmtree("c_generated_code", ignore_errors=True)

for f in glob.glob("acados_ocp_*.json"):
    os.remove(f)

for f in glob.glob("*.so"):
    os.remove(f)

print("Old ACADOS generated files removed.")

Old ACADOS generated files removed.


In [70]:
x_ref

NameError: name 'x_ref' is not defined

In [ ]:

# Initial condition
x0 = np.array([0.0, 0.0, np.deg2rad(-70.0), 0.0], dtype=float)

solver = create_ocp_solver(cfg, x0)

if solver is not None:
    steps = int(cfg.sim_time / cfg.dt)

    X = np.zeros((steps + 1, 4))
    U = np.zeros((steps, 1))
    X_REF = np.zeros((steps + 1, 4))
    U_REF = np.zeros((steps, 1))
    STATUS = np.zeros(steps, dtype=int)
    T = np.arange(steps + 1) * cfg.dt

    X[0] = x0

    for k in range(steps):
        t_now = k * cfg.dt
        x_ref_N = reference_at(t_now + cfg.N * cfg.dt, cfg)
        X_REF[k] = x_ref

        u0, status = solve_nmpc_step(solver, X[k], t_now, cfg)
        U[k] = u0
        STATUS[k] = status

        X[k + 1] = rk4_step(X[k], u0, cfg.dt, cfg)

    X_REF[-1] = reference_at(steps * cfg.dt, cfg)

    print("Done.")
    print("Unique ACADOS status values:", np.unique(STATUS))
else:
    print("Skip closed-loop simulation because ACADOS is not available.")


W shape      = (5, 5)
Vx shape     = (5, 4)
Vu shape     = (5, 1)
yref shape   = (5,)
W_e shape    = (4, 4)
Vx_e shape   = (4, 4)
yref_e shape = (4,)
External functions generated in 1.247 ms.
Templated solver code generated in 644.545 ms.
rm -f libacados_ocp_solver_rc_truck_reference_tracking.so
rm -f acados_solver_rc_truck_reference_tracking.o
cc -fPIC -std=c99   -O2 -I/home/juan/lehigh_PhD/monster_truck/NMPC/acados/include -I/home/juan/lehigh_PhD/monster_truck/NMPC/acados/include/acados -I/home/juan/lehigh_PhD/monster_truck/NMPC/acados/include/blasfeo/include -I/home/juan/lehigh_PhD/monster_truck/NMPC/acados/include/hpipm/include  -c -o acados_solver_rc_truck_reference_tracking.o acados_solver_rc_truck_reference_tracking.c
cc -fPIC -std=c99   -O2 -I/home/juan/lehigh_PhD/monster_truck/NMPC/acados/include -I/home/juan/lehigh_PhD/monster_truck/NMPC/acados/include/acados -I/home/juan/lehigh_PhD/monster_truck/NMPC/acados/include/blasfeo/include -I/home/juan/lehigh_PhD/monster_truck/NMPC/a

NameError: name 'x_ref' is not defined


## 6. Plot results


In [ ]:

if solver is not None:
    fig, axs = plt.subplots(5, 1, figsize=(12, 13), sharex=True)

    axs[0].plot(T, X[:, 0], label="p")
    axs[0].plot(T, X_REF[:, 0], "--", label="p_ref")
    axs[0].set_ylabel("position [m]")
    axs[0].grid(True)
    axs[0].legend()

    axs[1].plot(T, X[:, 1], label="v")
    axs[1].plot(T, X_REF[:, 1], "--", label="v_ref")
    axs[1].set_ylabel("velocity [m/s]")
    axs[1].grid(True)
    axs[1].legend()

    axs[2].plot(T, np.rad2deg(X[:, 2]), label="theta")
    axs[2].plot(T, np.rad2deg(X_REF[:, 2]), "--", label="theta_ref")
    axs[2].set_ylabel("pitch [deg]")
    axs[2].grid(True)
    axs[2].legend()

    axs[3].plot(T, X[:, 3], label="omega")
    axs[3].plot(T, X_REF[:, 3], "--", label="omega_ref")
    axs[3].set_ylabel("pitch rate [rad/s]")
    axs[3].grid(True)
    axs[3].legend()

    axs[4].step(T[:-1], U[:, 0], where="post", label="tau")
    axs[4].step(T[:-1], U_REF[:, 0], "--", where="post", label="tau_ref")
    axs[4].axhline(cfg.tau_min, linestyle=":", label="tau_min")
    axs[4].axhline(cfg.tau_max, linestyle=":", label="tau_max")
    axs[4].set_ylabel("torque [Nm]")
    axs[4].set_xlabel("time [s]")
    axs[4].grid(True)
    axs[4].legend()

    plt.tight_layout()
    plt.show()


In [ ]:

if solver is not None:
    plt.figure(figsize=(10, 6))
    plt.plot(X[:, 0], np.rad2deg(X[:, 2]), label="closed-loop trajectory")
    plt.plot(X_REF[:, 0], np.rad2deg(X_REF[:, 2]), "--", label="reference")
    plt.scatter([X[0, 0]], [np.rad2deg(X[0, 2])], label="start")
    plt.scatter([X[-1, 0]], [np.rad2deg(X[-1, 2])], label="end")
    plt.xlabel("position p [m]")
    plt.ylabel("pitch theta [deg]")
    plt.title("Phase plot: position vs pitch")
    plt.grid(True)
    plt.legend()
    plt.show()



## 7. How to adapt this notebook to your robot

Change these parts first:

1. **Dynamics:** edit `create_acados_model()` and `dynamics_np()` so the ACADOS model and plant simulation match.
2. **Reference:** edit `reference_at()`.
3. **Bounds:** edit the torque, velocity, angle, and angular-rate bounds in `TruckConfig`.
4. **Cost:** tune `Q`, `R`, and `Qf` using the fields in `TruckConfig`.

For a real robot, replace the RK4 plant step with your measured state update from ROS/MuJoCo/hardware:

```python
x_meas = read_state_from_robot()
u0, status = solve_nmpc_step(solver, x_meas, t_now, cfg)
send_torque_to_robot(u0[0])
```

A useful debugging rule: if the reference is not dynamically feasible, the optimizer will compromise. For this pitch model, setting both `theta_ref = 0` and `u_ref = 0` is not consistent with the pitch equation, because keeping a pitch equilibrium generally requires a nonzero torque.
